# Complainify AI — 04 : Multinomial Naive Bayes from Scratch

The classifier used for complaint auto-categorization, implemented without sklearn. Defines the class, trains on a sample, and makes predictions.


---
## PART 4: MULTINOMIAL NAIVE BAYES — FROM SCRATCH

### The Math

Bayes' Theorem:
```
P(category | text) ∝ P(category) × P(token₁ | category) × P(token₂ | category) × ...
```

Each word probability is smoothed with Laplace (add-1):
```
P(token | category) = (count + α) / (total_words + α × vocab_size)
```

We work in log-space to avoid underflow:
```
log P(category | text) = log P(category) + Σ log P(token | category)
```

In [1]:
import json, csv, os, math, re, random
from collections import Counter, defaultdict

# Category decoder: numeric ID to human name
CAT_DECODER = {0: 'IT Support', 1: 'Hostels', 2: 'Academics', 3: 'Fees / Finance', 4: 'Maintenance', 5: 'Transport',
               6: 'Security / Discipline', 7: 'Administration', 8: 'Library', 9: 'Canteen'}

In [2]:
STOPWORDS = set('''
    a an the is are was were be been being have has had do does did
    will would shall should may might must can could of in on at by
    for with about against between into through during before after
    above below to from up down out off over under again further then
    once here there when where why how all each every both few more
    most other some such no nor not only own same so than too very
    just because as until while
'''.split())

def stem(w):
    if len(w) < 5:
        return w
    if w.endswith('ingly'): return w[:-5]
    if w.endswith('edly'):  return w[:-4]
    if w.endswith('ying'):  return w[:-4] + 'y'
    if w.endswith('ation'): return w[:-5]
    if w.endswith('ment'):  return w[:-4]
    if w.endswith('able'):  return w[:-4]
    if w.endswith('ible'):  return w[:-4]
    if w.endswith('ness'):  return w[:-4]
    if w.endswith('less'):  return w[:-4]
    if w.endswith('ally'):  return w[:-4]
    if w.endswith('sion'):  return w[:-3] + 's'
    if w.endswith('tion'):  return w[:-3] + 't'
    if w.endswith('ical'):  return w[:-4]
    if w.endswith('ied'):   return w[:-3] + 'y'
    if w.endswith('ies'):   return w[:-3] + 'y'
    if w.endswith('ing'):   return w[:-3]
    if w.endswith('ive'):   return w[:-3]
    if w.endswith('ful'):   return w[:-3]
    if w.endswith('ous'):   return w[:-3]
    if w.endswith('ise'):   return w[:-3]
    if w.endswith('ize'):   return w[:-3]
    if w.endswith('ate'):   return w[:-3]
    if w.endswith('ify'):   return w[:-3]
    if w.endswith('ed'):    return w[:-2]
    if w.endswith('er'):    return w[:-2]
    if w.endswith('or'):    return w[:-2]
    if w.endswith('ly'):    return w[:-2]
    if w.endswith('al'):    return w[:-2]
    if w.endswith('en'):    return w[:-2]
    if w.endswith('s') and not w.endswith('ss'):
        return w[:-1]
    return w

def clean_and_tokenize(text, add_bigrams=True):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = [stem(t) for t in text.split() if t not in STOPWORDS and len(t) > 2]
    if add_bigrams and len(tokens) > 1:
        tokens += ['_'.join(pair) for pair in zip(tokens, tokens[1:])]
    return tokens

In [3]:
# Build the small teaching sample from the real dataset
import csv, random
rows = list(csv.DictReader(open(r'E:/Project-VI/workspace/ComplaintMgmtSystem/data/train_dataset.csv', encoding='utf-8')))
CAT_NAME_TO_ID = {v: k for k, v in CAT_DECODER.items()}
encoded = [{'text': r['text'], 'category_encoded': CAT_NAME_TO_ID[r['category']]}
           for r in rows if r['category'] in CAT_NAME_TO_ID]
random.seed(42)
sample_data = random.sample(encoded, 200)
print('sample_data size:', len(sample_data))


sample_data size: 200


In [4]:
class MultinomialNB:
    """
    Multinomial Naive Bayes classifier implemented from scratch.
    Uses Laplace smoothing and log-probabilities.
    """
    
    def __init__(self, alpha=1.0, min_df=3):
        self.alpha = alpha      # Laplace smoothing parameter
        self.min_df = min_df    # Ignore words appearing in fewer than min_df docs
        self._trained = False
    
    def fit(self, texts, labels):
        """
        TRAINING PHASE:
        
        What we learn:
        - self.classes:         All unique category IDs (e.g., [0, 1, 2, ..., 9])
        - self.priors:          log P(category) for each class
        - self.vocab:           Set of ~5,400 unique tokens (after min_df filter)
        - self.word_counts:     word_counts[class_id][token] = count
        - self.class_total_words: Total word count per class
        - self.vocab_size:      Size of vocabulary
        """
        self.classes = sorted(set(labels))
        n = len(texts)
        
        # --- Step 1: Compute class priors ---
        class_docs = Counter(labels)
        self.priors = {c: math.log(class_docs[c] / n) for c in self.classes}
        
        # --- Step 2: Tokenize all texts ---
        all_tokenized = [clean_and_tokenize(t) for t in texts]
        
        # --- Step 3: Build vocabulary (filter by min_df) ---
        doc_freq = Counter()
        for tokens in all_tokenized:
            for token in set(tokens):  # Count each doc once
                doc_freq[token] += 1
        self.vocab = {word for word, freq in doc_freq.items() if freq >= self.min_df}
        self.vocab_size = len(self.vocab)
        
        # --- Step 4: Count word occurrences per class ---
        self.word_counts = {c: defaultdict(int) for c in self.classes}
        self.class_total_words = {c: 0 for c in self.classes}
        
        for tokens, label in zip(all_tokenized, labels):
            for token in set(tokens):
                if token in self.vocab:
                    self.word_counts[label][token] += 1
                    self.class_total_words[label] += 1
        
        self._trained = True
        print(f'Model trained: {len(self.classes)} classes, {self.vocab_size} vocabulary')
    
    def predict_with_proba(self, text):
        """
        PREDICTION PHASE:
        
        For each class c:
            log_score(c) = log P(c) + sum(log P(token | c))
        
        Then convert log-scores to probabilities via softmax.
        """
        tokens = clean_and_tokenize(text)
        scores = {}
        
        for c in self.classes:
            log_prob = self.priors[c]
            total_wc = self.class_total_words[c]
            for token in tokens:
                count = self.word_counts[c].get(token, 0)
                # Laplace-smoothed: P(token|c) = (count + alpha) / (total + alpha * |V|)
                log_prob += math.log((count + self.alpha) / 
                                     (total_wc + self.alpha * self.vocab_size))
            scores[c] = log_prob
        
        # Softmax: convert log-scores to probabilities
        best = max(scores, key=scores.get)
        log_vals = list(scores.values())
        max_log = max(log_vals)
        exp_vals = [math.exp(v - max_log) for v in log_vals]
        total = sum(exp_vals)
        probs = {c: exp_vals[i] / total for i, c in enumerate(scores.keys())}
        
        return best, probs
    
    def save(self, path):
        """Save model to JSON (not pickle -- human-readable, safer)."""
        data = {
            'alpha': self.alpha,
            'min_df': self.min_df,
            'classes': self.classes,
            'priors': self.priors,
            'vocab': list(self.vocab),
            'vocab_size': self.vocab_size,
            'class_total_words': self.class_total_words,
            'word_counts': {str(c): dict(wc) for c, wc in self.word_counts.items()}
        }
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f'Model saved to {path}')
    
    @classmethod
    def load(cls, path):
        """Load model from JSON."""
        with open(path) as f:
            data = json.load(f)
        m = cls(alpha=data['alpha'], min_df=data['min_df'])
        m.classes = data['classes']
        m.priors = {int(k) if isinstance(k, str) and k.isdigit() else k: v 
                     for k, v in data['priors'].items()}
        m.vocab = set(data['vocab'])
        m.vocab_size = data['vocab_size']
        m.class_total_words = {int(k): v for k, v in data['class_total_words'].items()}
        m.word_counts = {
            int(c): defaultdict(int, {tok: cnt for tok, cnt in wc.items()})
            for c, wc in data['word_counts'].items()
        }
        m._trained = True
        return m


### Train the model on our sample data

In [5]:
# Train model on sample data
texts = [r['text'] for r in sample_data]
labels = [int(r['category_encoded']) for r in sample_data]

model = MultinomialNB(alpha=1.0, min_df=1)  # min_df=1 for small sample
model.fit(texts, labels)

print()
print('Vocabulary (first 10 tokens):', list(model.vocab)[:10])

Model trained: 10 classes, 1759 vocabulary

Vocabulary (first 10 tokens): ['blast_already', 'key_roomm', 'laundry', 'mud_hallway', 'week', 'key', 'cab', 'fund_refund', 'line_mass', 'evidence_laptop']


### Make predictions

In [6]:
def predict(text):
    pred, probs = model.predict_with_proba(text)
    print(f'Text: "{text}"')
    print(f'Predicted: {CAT_DECODER[pred]} (ID: {pred})')
    print(f'Confidence: {probs[pred]:.2%}')
    print('All probabilities:')
    for cid in sorted(probs.keys()):
        print(f'  {CAT_DECODER[cid]:20s}: {probs[cid]:.2%}')
    print()

predict('wifi is slow in the library')
predict('canteen food is very bad quality')

Text: "wifi is slow in the library"
Predicted: IT Support (ID: 0)
Confidence: 41.56%
All probabilities:
  IT Support          : 41.56%
  Hostels             : 11.61%
  Academics           : 3.61%
  Fees / Finance      : 5.98%
  Maintenance         : 4.20%
  Transport           : 4.80%
  Security / Discipline: 5.03%
  Administration      : 12.13%
  Library             : 5.35%
  Canteen             : 5.72%

Text: "canteen food is very bad quality"
Predicted: Canteen (ID: 9)
Confidence: 34.64%
All probabilities:
  IT Support          : 7.41%
  Hostels             : 7.92%
  Academics           : 4.74%
  Fees / Finance      : 9.02%
  Maintenance         : 5.74%
  Transport           : 6.77%
  Security / Discipline: 6.82%
  Administration      : 8.70%
  Library             : 8.24%
  Canteen             : 34.64%

